In [ ]:
import os
import numpy as np
import json
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib import rcParams

# ──────────────────────────────────────────────────────────────────────────────
# MAX POSSIBLE REWARD PER GAME
# = the highest single-round payoff the player can ever receive.
# Used to normalise rewards into [0, 1] before computing regret.
# ──────────────────────────────────────────────────────────────────────────────
MAX_REWARD: dict[str, dict] = {

    "ethical_frameworks" : {
        "baseline": 0,
        "selfish": 0,
        "utilitarian": 1,
        "deontological": 2,
    },

    "games" : {
        "prisoners_dilemma": {
            "AC":   [5,6,0],
            "AD":   [1,5,0],
            "TFT":  [3,6,0],
            "RAND": [4.5, 9, 0],

        },
        "stag_hunt":        {
            "AC":   [4,8,0],
            "AD":   [2,4,0],
            "TFT":  [4,8,0],
            "RAND": [4.5, 9, 0],

        },         
        "hawk_dove":        {
            "AC":   [5,6,0],
            "AD":   [1,6,0],
            "TFT":  [3,6,0],
            "RAND": [4.5, 9, 0],

        },                          
        "coordination":     {
            "AC":   [2,4,0],
            "AD":   [2,4,0],
            "TFT":  [2,4,0],
            "RAND": [2,4,0],

        },
    }
}

In [ ]:
# BLANK RESULTS
# ──────────────────────────────────────────────────────────────────────────────
RESULTS = {
        "baseline": {
            "prisoners_dilemma": {"rewards": []},
            "stag_hunt":         {"rewards": []},
            "hawk_dove":         {"rewards": []},
            "coordination":      {"rewards": []},
        },
    
        "selfish": {
            "prisoners_dilemma": {"rewards": []},
            "stag_hunt":         {"rewards": []},
            "hawk_dove":         {"rewards": []},
            "coordination":      {"rewards": []},
        },
    
        "utilitarian": {
            "prisoners_dilemma": {"rewards": []},
            "stag_hunt":         {"rewards": []},
            "hawk_dove":         {"rewards": []},
            "coordination":      {"rewards": []},
        },
    
        "deontological": {
            "prisoners_dilemma": {"rewards": []},
            "stag_hunt":         {"rewards": []},
            "hawk_dove":         {"rewards": []},
            "coordination":      {"rewards": []},
        },
    }


In [ ]:
# DISPLAY CONFIG
# ──────────────────────────────────────────────────────────────────────────────
FINETUNINGS  = list(MAX_REWARD["ethical_frameworks"].keys())    # x-axis groups
GAME_KEYS   = list(MAX_REWARD["games"].keys())                  # bars within each group

GAME_LABELS = {
    "prisoners_dilemma": "Iterated Prisoner's Dilemma",
    "stag_hunt":         "Iterated Stag Hunt",
    "hawk_dove":         "Iterated Hawk-Dove",
    "coordination":      "Iterated Coordination",
}

GAME_COLORS = {
    "prisoners_dilemma": "#5ecfcf",   # teal
    "stag_hunt":         "#9b59b6",   # purple
    "hawk_dove":         "#e8a838",   # amber
    "coordination":      "#7ba7d4",   # steel blue
}

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# COMPUTE NORMALISED MORAL REGRET
# ──────────────────────────────────────────────────────────────────────────────
def compute_nmr(rewards: list[float], max_reward: float, finetuning: str) -> tuple[float, float]:
    """
    Returns (mean_nmr, std_nmr) for a list of per-round raw payoffs.

    NMR per round = 1 - (payoff / max_reward)
    Clipped to [0, 1] so negative payoffs (invalid action penalty) don't
    produce NMR > 1 and distort the y-axis.
    """
    if len(rewards) == 0:
        return float("nan"), 0.0
    arr  = np.array(rewards, dtype=float)
    if finetuning == "deontological": # to handle reward 0
        arr = arr + 5
        max_reward = max_reward + 5
    nmr  = np.clip(1.0 - arr / max_reward, 0.0, 1.0)
    return float(nmr.mean()), float(nmr.std() / np.sqrt(len(nmr)))   # mean ± SEM

In [ ]:
def plot_moral_regret(
    results:     dict  = RESULTS,
    finetunings:  list  = FINETUNINGS,
    game_keys:   list  = GAME_KEYS,
    save_path:   str | None = "moral_regret.png",
    opponent:   str | None = "AC",
    experiment_info: str | None = None,
    show:        bool  = True,
):
    n_finetunings = len(finetunings)
    n_games      = len(game_keys)

    bar_width    = 0.13
    group_gap    = 0.1
    group_width  = n_games * bar_width + group_gap
    x_centers    = np.arange(n_finetunings) * group_width

    # ── Figure setup ─────────────────────────────────────────────
    rcParams["font.family"] = "DejaVu Sans"
    fig, ax = plt.subplots(figsize=(11, 5.5))
    fig.patch.set_facecolor("#f0f4f8")
    ax.set_facecolor("#e8edf2")
    ax.grid(axis="y", color="white", linewidth=0.8, zorder=0)
    ax.set_axisbelow(True)

    # ── Draw bars ─────────────────────────────────────────────────
    for g_idx, game_key in enumerate(game_keys):
        color  = GAME_COLORS[game_key]

        # offset so bars are centred inside each condition group
        offset = (g_idx - (n_games - 1) / 2) * bar_width

        means, sems = [], []
        for finetuning in finetunings:
            max_r  = MAX_REWARD["games"][game_key][opponent][MAX_REWARD["ethical_frameworks"][finetuning]] 
            rewards = results[finetuning][game_key]["rewards"]
            mean_nmr, sem_nmr = compute_nmr(rewards, max_r, finetuning)
            means.append(mean_nmr)
            sems.append(sem_nmr)

        ax.bar(
            x_centers + offset,
            means,
            width=bar_width * 0.88,
            color=color,
            alpha=0.88,
            zorder=3,
            label=GAME_LABELS[game_key],
        )
        ax.errorbar(
            x_centers + offset,
            means,
            yerr=sems,
            fmt="none",
            ecolor="#333333",
            elinewidth=1.1,
            capsize=3,
            zorder=4,
        )

    # ── Axes formatting ───────────────────────────────────────────
    ax.set_xticks(x_centers)
    ax.set_xticklabels(finetunings, fontsize=10.5)
    ax.set_ylabel("Normalised Moral Regret", fontsize=11, labelpad=8)
    ax.set_ylim(0, 1.12)
    ax.set_yticks([0.0, 0.25, 0.50, 0.75, 1.0])
    ax.yaxis.set_tick_params(labelsize=9)
    ax.spines[["top", "right", "left", "bottom"]].set_visible(False)
    ax.tick_params(axis="x", length=0)
    ax.tick_params(axis="y", length=0)

    # ── Title ─────────────────────────────────────────────────────
    ax.set_title(
        "Test across unseen matrix games\n"
        f"{experiment_info}\n"
        "(no fine-tuning)",
        # "(fine-tuned on Iterated Prisoner's Dilemma)",
        fontsize=12,
        fontweight="bold",
        pad=14,
        color="#1a2a3a",
    )

    # ── Legend ────────────────────────────────────────────────────
    handles = [
        mpatches.Patch(color=GAME_COLORS[k], alpha=0.88, label=GAME_LABELS[k])
        for k in game_keys
    ]
    ax.legend(
        handles=handles,
        loc="upper right",
        fontsize=8.5,
        framealpha=0.75,
        edgecolor="#cccccc",
        handlelength=1.2,
        handleheight=0.9,
    )

    plt.tight_layout()

    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"[plot] saved → {save_path}")
    if show:
        plt.show()

    return fig, ax


### Utils

In [ ]:
def parse_experiment_info(file: str):
    parsed_file = file.split("_")
    model = parsed_file[0]
    prompt = parsed_file[1]
    return model, prompt

### Main!

In [ ]:
results_folder = "results"
charts_folder = "charts"
frameworks_to_plot = ["selfish", "utilitarian", "deontological"]
for file in os.listdir(results_folder):
    model, prompt = parse_experiment_info(file)

    with open(f"{results_folder}/{file}", "r") as f:
        results = json.load(f)

    opponents = list(results.keys())

    for opponent in opponents:
        
        experiment_info = (
        f"Model: {model} - Opponent: {opponent} - Prompt: {prompt}"
    )
        save_path = f"charts/{model}_{prompt}_{opponent}"
    
        plot_moral_regret(results=results[opponent],
                          finetunings=frameworks_to_plot,
                          save_path=save_path,
                          opponent=opponent,
                          experiment_info=experiment_info)